In [ ]:
# =============================================================================
# Acquirung training data for optimization pipeline
# =============================================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
import joblib
import glob
import os
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

# ── Configuration — must match the main training script ───────────────────────
DATA_DIR = r'C:\Users\lambe\RU_Thesis_2026\ml_data'

DYNAMIC_FEATURES = [
    'temp_max', 'rh_min', 'vpd_mean', 'wind_speed_mean',
    'precip_sum', 'solar_rad_mj', 'rh_7d', 'temp_7d',
    'precip_30d', 'recovery_value'
]
STATIC_FEATURES = [
    'elevation_m', 'slope_deg', 'land_cover',
    'pop_density', 'road_proximity_m'
]
FEATURES       = DYNAMIC_FEATURES + STATIC_FEATURES
TRAIN_YEARS    = list(range(2010, 2021))
VAL_YEARS      = [2021, 2022]
DOY_START_HARD = 91
DOY_END_HARD   = 310

# ── Load training and validation data ─────────────────────────────────────────
def load_years(years, split_label):
    all_dfs = []
    for year in years:
        pattern = os.path.join(DATA_DIR, f'ML_{split_label}_{year}*.csv')
        for f in sorted(glob.glob(pattern)):
            all_dfs.append(pd.read_csv(f))
    if not all_dfs:
        raise FileNotFoundError(f"No CSV files for {split_label} {years}")
    return pd.concat(all_dfs, ignore_index=True)

def clean_and_filter(df):
    required = FEATURES + ['target_ignition', 'fire_cause', 'date']
    df = df.dropna(subset=required)
    df['target_ignition'] = df['target_ignition'].astype(int)
    df['fire_cause']      = df['fire_cause'].astype(int)
    df['land_cover']      = df['land_cover'].astype(int)
    df = df[df['target_ignition'].isin([0, 1])]
    df = df[df['fire_cause'].between(0, 5)]
    df['date_parsed'] = pd.to_datetime(df['date'], format='%Y%m%d')
    df['doy']         = df['date_parsed'].dt.dayofyear
    df = df[(df['doy'] >= DOY_START_HARD) &
            (df['doy'] <= DOY_END_HARD)].copy()
    return df.drop(columns=['date_parsed', 'doy'])

print("Loading training data...")
df_train_raw = load_years(TRAIN_YEARS, 'train')
df_train     = clean_and_filter(df_train_raw)
df_train     = df_train.sort_values('date').reset_index(drop=True)
print(f"  Train rows: {len(df_train):,}")

print("Loading validation data...")
df_val_raw = load_years(VAL_YEARS, 'val')
df_val     = clean_and_filter(df_val_raw)
df_val     = df_val.sort_values('date').reset_index(drop=True)
print(f"  Val rows  : {len(df_val):,}")

# ── Prepare arrays ────────────────────────────────────────────────────────────
X_train     = df_train[FEATURES].values
y_ign_train = df_train['target_ignition'].values
X_val       = df_val[FEATURES].values
y_ign_val   = df_val['target_ignition'].values

fire_tr_mask = df_train['target_ignition'] == 1
n_pos        = fire_tr_mask.sum()
n_neg        = (~fire_tr_mask).sum()
ratio_tr     = n_neg / max(n_pos, 1)

print(f"\nClass balance:")
print(f"  Fire    : {n_pos:,}")
print(f"  No-fire : {n_neg:,}")
print(f"  Ratio   : {ratio_tr:.0f}:1")

# Load CV threshold from previous run
threshold_record = joblib.load('models/xgb_cv_threshold.pkl')
CV_THRESHOLD     = threshold_record['cv_threshold']
print(f"  CV threshold: {CV_THRESHOLD:.4f}")

# ── Run the tuning subsample block ────────────────────────────────────────
TUNE_FIRE_RATIO = 50

n_fire_tune   = fire_tr_mask.sum()
n_nofire_tune = int(n_fire_tune * TUNE_FIRE_RATIO)

nofire_tune_idx = np.random.RandomState(0).choice(
    np.where(~fire_tr_mask.values)[0], size=n_nofire_tune, replace=False)
tune_idx = np.concatenate([np.where(fire_tr_mask.values)[0],
                            nofire_tune_idx])
np.random.RandomState(0).shuffle(tune_idx)

X_tune = X_train[tune_idx]
y_tune = y_ign_train[tune_idx]

print(f"\nTuning sample: {len(X_tune):,} rows  "
      f"({y_tune.sum():,} fire + {(y_tune==0).sum():,} no-fire)")



In [ ]:
# ── Helper: print model hyperparameters ───────────────────────────────────────

def print_params(model, label):
    """Print key hyperparameters of an XGBoost model."""
    params = model.get_params()

    key_params = ['objective', 'n_estimators', 'max_depth',
                  'learning_rate', 'subsample', 'colsample_bytree',
                  'min_child_weight', 'gamma', 'reg_alpha', 'reg_lambda',
                  'scale_pos_weight', 'eval_metric',
                  'early_stopping_rounds', 'random_state']

    print(f"\n  Parameters — {label}:")
    print(f"  {'-'*50}")
    for k in key_params:
        if k in params and params[k] is not None:
            v = params[k]
            if isinstance(v, float):
                print(f"    {k:<25} {v:.6f}")
            else:
                print(f"    {k:<25} {v}")
    print(f"  {'-'*50}")

In [ ]:
# =============================================================================
# XGBoost Ignition Model — Optuna Optimisation
# WITH per-step evaluation for thesis documentation
# =============================================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
import joblib
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score,
                             classification_report, confusion_matrix,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold
import time
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
MIN_RECALL     = 0.70
N_FOLDS_OPTUNA = 3
N_FOLDS_THRESH = 5
N_TRIALS       = 50
RANDOM_STATE   = 42

# ── Helper: print full evaluation at a given threshold ────────────────────────
def evaluate_model(model, X, y_true, threshold, label):
    """Print full validation metrics at a given threshold."""
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    print(f"\n{'='*60}")
    print(f"EVALUATION: {label}")
    print(f"  Threshold         : {threshold:.4f}")
    print(f"{'='*60}")
    print(f"  ROC-AUC           : {roc_auc_score(y_true, y_prob):.4f}")
    print(f"  Balanced accuracy : {balanced_accuracy_score(y_true, y_pred):.4f}")
    print(f"  Recall (fire)     : {tp/(tp+fn):.4f}  "
          f"({tp} of {tp+fn} fires detected)")
    print(f"  Precision (fire)  : {tp/(tp+fp) if (tp+fp)>0 else 0:.4f}")
    print(f"  FPR               : {fp/(fp+tn):.4f}  "
          f"({fp/(fp+tn)*100:.1f}% no-fire days flagged)")
    print(f"  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")
    print(classification_report(y_true, y_pred,
                                target_names=['No fire', 'Fire'],
                                digits=4, zero_division=0))
    return {
        'label'    : label,
        'threshold': threshold,
        'auc'      : roc_auc_score(y_true, y_prob),
        'recall'   : tp/(tp+fn),
        'precision': tp/(tp+fp) if (tp+fp) > 0 else 0,
        'fpr'      : fp/(fp+tn),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
    }

# ── Store results for final comparison table ──────────────────────────────────
stage_results = []

# =============================================================================
# BASELINE: Original manually-tuned model (no Optuna)
# =============================================================================
print("\n" + "="*60)
print("BASELINE: Loading original manually-tuned model")
print("="*60)

try:
    bundle         = joblib.load('models/model_ignition_2010_2022_with_threshold.pkl')
    model_baseline = bundle['model']
    CV_THRESHOLD_BASELINE = bundle['cv_threshold']
    print(f"  Loaded: model_ignition_2010_2022_with_threshold.pkl")
    print(f"  CV threshold from previous run: {CV_THRESHOLD_BASELINE:.4f}")

    # Evaluate at default threshold
    r = evaluate_model(model_baseline, X_val, y_ign_val,
                       0.5, 'Baseline — default threshold (0.50)')
    stage_results.append(r)

    # Evaluate at CV-optimised threshold
    r = evaluate_model(model_baseline, X_val, y_ign_val,
                       CV_THRESHOLD_BASELINE,
                       f'Baseline — CV threshold ({CV_THRESHOLD_BASELINE:.3f})')
    stage_results.append(r)

except FileNotFoundError:
    print("  Baseline model not found — training fresh baseline...")

    model_baseline = xgb.XGBClassifier(
        objective             = 'binary:logistic',
        n_estimators          = 500,
        max_depth             = 5,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        min_child_weight      = 10,
        scale_pos_weight      = ratio_tr,
        eval_metric           = 'auc',
        early_stopping_rounds = 20,
        random_state          = RANDOM_STATE,
        n_jobs                = -1,
        verbosity             = 1
    )
    model_baseline.fit(X_train, y_ign_train,
                       eval_set=[(X_val, y_ign_val)], verbose=50)

    print_params(model_baseline, 'Baseline')  

    r = evaluate_model(model_baseline, X_val, y_ign_val,
                       0.5, 'Baseline — default threshold (0.50)')
    stage_results.append(r)

# =============================================================================
# STAGE 1: Optuna tuning 
# =============================================================================
print("\n" + "="*60)
print("STAGE 1: Optuna tuning")
print("="*60)

def objective_optuna(trial):
    params = {
        'objective'            : 'binary:logistic',
        'eval_metric'          : 'auc',
        'scale_pos_weight'     : ratio_tr,
        'n_estimators'         : trial.suggest_int('n_estimators', 100, 800),
        'max_depth'            : trial.suggest_int('max_depth', 3, 8),
        'learning_rate'        : trial.suggest_float('learning_rate',
                                                      0.01, 0.3, log=True),
        'subsample'            : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'     : trial.suggest_float('colsample_bytree',
                                                      0.5, 1.0),
        'min_child_weight'     : trial.suggest_int('min_child_weight', 1, 30),
        'gamma'                : trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha'            : trial.suggest_float('reg_alpha',
                                                      1e-4, 10.0, log=True),
        'reg_lambda'           : trial.suggest_float('reg_lambda',
                                                      1e-4, 10.0, log=True),
        'random_state'         : RANDOM_STATE,
        'n_jobs'               : -1,
        'verbosity'            : 0,
        'early_stopping_rounds': 15,
    }

    skf  = StratifiedKFold(n_splits=N_FOLDS_OPTUNA,
                           shuffle=True, random_state=RANDOM_STATE)
    aucs = []
    for tr_idx, vl_idx in skf.split(X_train, y_ign_train):
        n_est = params.pop('n_estimators')
        early = params.pop('early_stopping_rounds')
        m = xgb.XGBClassifier(n_estimators=n_est,
                               early_stopping_rounds=early, **params)
        m.fit(X_train[tr_idx], y_ign_train[tr_idx],
              eval_set=[(X_train[vl_idx], y_ign_train[vl_idx])],
              verbose=False)
        aucs.append(roc_auc_score(y_ign_train[vl_idx],
                                   m.predict_proba(X_train[vl_idx])[:, 1]))
        params['n_estimators']          = n_est
        params['early_stopping_rounds'] = early
    return np.mean(aucs)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study1 = optuna.create_study(direction='maximize',
                              sampler=optuna.samplers.TPESampler(
                                  seed=RANDOM_STATE))
study1.optimize(objective_optuna, n_trials=N_TRIALS,
                timeout=28800, show_progress_bar=True)

best_params_optuna = study1.best_params
print(f"\n  Best CV ROC-AUC (Optuna): {study1.best_value:.4f}")

print(f"\n  Optuna best parameters:")
for k, v in best_params_optuna.items():
    if isinstance(v, float):
        print(f"    {k:<25} {v:.6f}")
    else:
        print(f"    {k:<25} {v}")

# Retrain on full data with best params
t0 = time.time()
model_optuna_only = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='auc',
    scale_pos_weight=ratio_tr, early_stopping_rounds=20,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=1,
    **best_params_optuna)
model_optuna_only.fit(X_train, y_ign_train,
                      eval_set=[(X_val, y_ign_val)], verbose=50)
print(f"  Training time: {(time.time()-t0)/60:.1f} minutes")

print_params(model_optuna_only, 'Optuna')

# Evaluate at default threshold
r = evaluate_model(model_optuna_only, X_val, y_ign_val,
                   0.5, 'Optuna — default threshold (0.50)')
stage_results.append(r)


# Run threshold CV on this model
print("\n  Running threshold CV on Optuna model...")
fold_thresholds_1 = []
skf_t = StratifiedKFold(n_splits=N_FOLDS_THRESH,
                         shuffle=True, random_state=RANDOM_STATE)
for tr_idx, vl_idx in skf_t.split(X_train, y_ign_train):
    y_prob_f = model_optuna_only.predict_proba(X_train[vl_idx])[:, 1]
    candidates = np.linspace(0.01, 0.99, 200)
    best_thr, best_prec = 0.5, 0.0
    for thr in candidates:
        y_t = (y_prob_f >= thr).astype(int)
        rec  = recall_score(y_ign_train[vl_idx], y_t, zero_division=0)
        prec = precision_score(y_ign_train[vl_idx], y_t, zero_division=0)
        if rec >= MIN_RECALL and prec > best_prec:
            best_prec, best_thr = prec, thr
    fold_thresholds_1.append(best_thr)

cv_thr_1 = float(np.mean(fold_thresholds_1))
print(f"  CV threshold (Optuna): {cv_thr_1:.4f}")

r = evaluate_model(model_optuna_only, X_val, y_ign_val,
                   cv_thr_1,
                   f'Optuna — CV threshold ({cv_thr_1:.3f})')
stage_results.append(r)

# =============================================================================
# SUMMARY TABLE — all stages
# =============================================================================
print("\n" + "="*70)
print("OPTIMISATION PIPELINE — SUMMARY OF ALL STAGES")
print("="*70)
print(f"  {'Stage':<45} {'AUC':>7} {'Recall':>8} {'FPR':>8} "
      f"{'TP':>7} {'FP':>10}")
print(f"  {'-'*85}")
for r in stage_results:
    print(f"  {r['label']:<45} {r['auc']:>7.4f} {r['recall']:>8.4f} "
          f"{r['fpr']:>8.4f} {r['tp']:>7,} {r['fp']:>10,}")

# =============================================================================
# Save final model
# =============================================================================
joblib.dump({'model'            : model_optuna_only,
             'cv_threshold'     : cv_thr_1,
             'features'         : FEATURES,
             'best_params'      : best_params_optuna,
             'optuna_auc'       : study1.best_value,
             'train_years'      : TRAIN_YEARS,
             'val_years'        : VAL_YEARS},
            'models/model_ignition_final.pkl')

print("\nSaved: models/model_ignition_final.pkl")


# ── Parameter comparison across stages ────────────────────────────────────────
print(f"\n{'='*70}")
print("IGNITION MODEL — PARAMETER COMPARISON ACROSS STAGES")
print(f"{'='*70}")

compare_params = ['n_estimators', 'max_depth', 'learning_rate',
                  'subsample', 'colsample_bytree', 'min_child_weight',
                  'gamma', 'reg_alpha', 'reg_lambda', 'scale_pos_weight']

models_named = [
    ('Baseline',           model_baseline),
    ('Optuna',   model_optuna_only)
]

# Header row
header = f"  {'Parameter':<25}"
for name, _ in models_named:
    header += f" {name:>18}"
print(header)
print(f"  {'-'*80}")

for param in compare_params:
    row = f"  {param:<25}"
    for name, model in models_named:
        v = model.get_params().get(param, 'N/A')
        if isinstance(v, float):
            row += f" {v:>18.4f}"
        elif isinstance(v, int):
            row += f" {v:>18d}"
        else:
            row += f" {str(v):>18}"
    print(row)

# ── CV threshold comparison ───────────────────────────────────────────────────
print(f"\n  {'CV threshold':<25} {'N/A':>18} "
      f"{cv_thr_1:>18.4f} {CV_THRESHOLD:>18.4f}")

print(f"\n{'='*70}")
print("PERFORMANCE + PARAMETER COMBINED SUMMARY")
print(f"{'='*70}")
print(f"  {'Stage':<45} {'AUC':>7} {'Recall':>8} "
      f"{'FPR':>8} {'TP':>7} {'FP':>10}")
print(f"  {'-'*85}")
for r in stage_results:
    print(f"  {r['label']:<45} {r['auc']:>7.4f} "
          f"{r['recall']:>8.4f} {r['fpr']:>8.4f} "
          f"{r['tp']:>7,} {r['fp']:>10,}")

